Fine-tune FinBERT on 12K synthetic transaction descriptions to classify into 12 spending categories.


In [1]:
!pip install -q transformers datasets torch scikit-learn pandas numpy loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00


In [2]:
# upload data
from google.colab import files
import os

print("Upload Data")
uploaded = files.upload()

# Verify upload
if 'transaction_labels.csv' in uploaded:
    print("Data uploaded successfully")
    csv_path = 'transaction_labels.csv'
else:
    print("Error")
    csv_path = None

Upload Data


Saving transaction_labels.csv to transaction_labels.csv
Data uploaded successfully


In [3]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import json

# Load data
df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

# Category distribution
print(f"\nCategory Distribution:")
print(df['category'].value_counts())

# Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())

Shape: (12000, 2)

Columns: ['description', 'category']

First 5 rows:
                description                 category
0       SQ*MUTUAL FUND 3350               Investment
1               SALARY #697  Income / Direct Deposit
2            ATM WITHDRAWAL                 Transfer
3  ATM WITHDRAWAL STORE #32                 Transfer
4        WIRE TRANSFER 4438                 Transfer

Category Distribution:
category
Investment                 1000
Income / Direct Deposit    1000
Transfer                   1000
Healthcare                 1000
Utilities                  1000
Food and Dining            1000
Other                      1000
Shopping                   1000
Insurance                  1000
Refund                     1000
Travel                     1000
Entertainment              1000
Name: count, dtype: int64

Missing values:
description    0
category       0
dtype: int64


In [4]:
# Data Categories and Data Split
from sklearn.model_selection import train_test_split

CATEGORIES = [
    "Food and Dining",
    "Shopping",
    "Travel",
    "Healthcare",
    "Utilities",
    "Income / Direct Deposit",
    "Transfer",
    "Refund",
    "Entertainment",
    "Insurance",
    "Investment",
    "Other"
]

# Create label mapping
label2id = {label: i for i, label in enumerate(CATEGORIES)}
id2label = {i: label for i, label in enumerate(CATEGORIES)}

# Map categories to labels
df['label'] = df['category'].map(label2id)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(f"Total samples after label mapping: {len(df)}")
print(f"Label distribution: {df['label'].value_counts().sort_index().to_dict()}")

# 80/10/10 split with stratification
train_val, test = train_test_split(
    df, test_size=0.1, random_state=42, stratify=df['label']
)
train, val = train_test_split(
    train_val, test_size=0.1/(1-0.1), random_state=42, stratify=train_val['label']
)

print(f"\nData Split:")
print(f"  Train: {len(train):,}")
print(f"  Val: {len(val):,}")
print(f"  Test: {len(test):,}")

Total samples after label mapping: 12000
Label distribution: {0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000, 5: 1000, 6: 1000, 7: 1000, 8: 1000, 9: 1000, 10: 1000, 11: 1000}

Data Split:
  Train: 9,600
  Val: 1,200
  Test: 1,200


In [5]:
# loading model and Tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "ProsusAI/finbert"
max_length = 64

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(CATEGORIES),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print(f"Model loaded")
print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Device: {next(model.parameters()).device}")

Loading ProsusAI/finbert...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |                                                                                      
-----------------------------+------------+--------------------------------------------------------------------------------------
bert.embeddings.position_ids | UNEXPECTED |                                                                                      
classifier.weight            | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([12, 768])
classifier.bias              | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([12])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Model loaded
  Params: 109,491,468
  Device: cpu


In [6]:
# create pytorch dataset
from torch.utils.data import Dataset, DataLoader

class TransactionDataset(Dataset):
    """Custom dataset for transaction descriptions."""

    def __init__(self, descriptions, labels, tokenizer, max_length=64):
        self.descriptions = descriptions
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        text = self.descriptions[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = TransactionDataset(
    train['description'].values,
    train['label'].values,
    tokenizer,
    max_length
)
val_dataset = TransactionDataset(
    val['description'].values,
    val['label'].values,
    tokenizer,
    max_length
)
test_dataset = TransactionDataset(
    test['description'].values,
    test['label'].values,
    tokenizer,
    max_length
)

print(f"  Train: {len(train_dataset)}")
print(f"  Val: {len(val_dataset)}")
print(f"  Test: {len(test_dataset)}")

  Train: 9600
  Val: 1200
  Test: 1200


## Training Setup

**Anti-Overfitting Measures:**
- Early stopping with patience=2 (stops if val_loss doesn't improve for 2 epochs)
- Weight decay (L2 regularization) = 0.01

**Anti-Catastrophic Forgetting Measures:**
- Low learning rate (2e-5) for fine-tuning
- Learning rate scheduler (cosine with warmup) to gradually decay LR
- Warmup steps (100) to stabilize initial training
- Only 5 epochs to avoid excessive adaptation


In [7]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01

print(f"Hyperparameters:")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Warmup Steps: {WARMUP_STEPS}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  Early Stopping Patience: 2")

# Define compute_metrics function
def compute_metrics(eval_pred):
    """Compute metrics for validation."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)
    precision, recall, _, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision': precision,
        'recall': recall
    }

print("Metrics function ready")

Hyperparameters:
  Batch Size: 32
  Learning Rate: 2e-05
  Epochs: 5
  Warmup Steps: 100
  Weight Decay: 0.01
  Early Stopping Patience: 2
Metrics function ready


In [10]:
# training

import time

# Training arguments
training_args = TrainingArguments(
    output_dir='./training_outputs',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to=[],  # Disable wandb
    gradient_accumulation_steps=1,
    optim='adamw_torch'  # Memory-efficient optimizer
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Train
print("\nStarting training...")
print("=" * 70)
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
print("=" * 70)
print(f"\nTraining Time: {elapsed/60:.2f} minutes")


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision,Recall
1,0.041178,0.033008,0.993333,0.993323,0.993323,0.993827,0.993333
2,0.018156,0.013323,0.995833,0.995831,0.995831,0.996032,0.995833
3,0.011979,0.014349,0.993333,0.993323,0.993323,0.993827,0.993333
4,0.011505,0.011507,0.993333,0.993323,0.993323,0.993827,0.993333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Training Time: 9.68 minutes


In [12]:
# Evaluating on test set
test_results = trainer.evaluate(test_dataset, metric_key_prefix='test')

print("\n" + "=" * 70)
print("TEST SET EVALUATION")
print("=" * 70)
print(f"\nAccuracy:        {test_results.get('test_accuracy', 0):.4f}")
print(f"F1 (Macro):      {test_results.get('test_f1_macro', 0):.4f}")
print(f"F1 (Weighted):   {test_results.get('test_f1_weighted', 0):.4f}")
print(f"Precision:       {test_results.get('test_precision', 0):.4f}")
print(f"Recall:          {test_results.get('test_recall', 0):.4f}")
print("\n" + "=" * 70)

# Check if meets success criteria
accuracy = test_results.get('test_accuracy', 0)
f1_macro = test_results.get('test_f1_macro', 0)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1: {f1_macro:.4f}")

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled



TEST SET EVALUATION

Accuracy:        0.9958
F1 (Macro):      0.9958
F1 (Weighted):   0.9958
Precision:       0.9960
Recall:          0.9958

Accuracy: 0.9958
F1: 0.9958


In [13]:
from sklearn.metrics import confusion_matrix, classification_report

# Get predictions
print("Computing per-class metrics...")
test_predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(test_predictions.predictions, axis=1)
true_labels = test_dataset.labels

print(f"\nPer-Class Metrics (F1 Score):")
print(classification_report(
    true_labels, pred_labels,
    target_names=CATEGORIES,
    digits=4
))
print(confusion_matrix(true_labels, pred_labels))

Computing per-class metrics...



Per-Class Metrics (F1 Score):
                         precision    recall  f1-score   support

        Food and Dining     0.9524    1.0000    0.9756       100
               Shopping     1.0000    0.9500    0.9744       100
                 Travel     1.0000    1.0000    1.0000       100
             Healthcare     1.0000    1.0000    1.0000       100
              Utilities     1.0000    1.0000    1.0000       100
Income / Direct Deposit     1.0000    1.0000    1.0000       100
               Transfer     1.0000    1.0000    1.0000       100
                 Refund     1.0000    1.0000    1.0000       100
          Entertainment     1.0000    1.0000    1.0000       100
              Insurance     1.0000    1.0000    1.0000       100
             Investment     1.0000    1.0000    1.0000       100
                  Other     1.0000    1.0000    1.0000       100

               accuracy                         0.9958      1200
              macro avg     0.9960    0.9958    0.9958   

In [14]:
import shutil

# Save model
save_path = "finbert-transaction"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved
